# Practical identifiability — could a trial even estimate this parameter? (design axis)

**NOT FOR CLINICAL USE.** Population/design-level only; no individual quantity, no prognosis, no therapy ranking. Executed in CI (nbmake) so the result cannot silently break.

Onkos surfaces that the kill/resistance terms carry ~90% IIV CV, with the stated reason that *resistance is poorly identifiable from short trials*. This notebook **measures** that claim: from the Fisher information of a realistic scan schedule it computes the Cramér-Rao lower bound on each structural parameter's precision, and flags the parameters a trial of that shape cannot pin down — and the CVs that are therefore partly flat-likelihood artifacts. See `onkos.identify` and `docs/specs/research/practical-identifiability.md`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos

ds = onkos.load()
print(f'{len(ds)} records | onkos {onkos.__version__}')

## The verdict for the canonical clinical TGI model

The Claret NSCLC model under a realistic RECIST cadence (scans at weeks 0, 6, 12, 18, 24, 36, 48; 20% proportional assay error). Each parameter's *predicted* RSE sits next to its *stored* IIV CV.

In [ ]:
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
res = onkos.identifiability(ds, 'resistance.claret_2009.tgi', context=ctx)

print(f'tier {res.tier}  |  collinearity index gamma_K = {res.collinearity_index:.1f}'
      f'  |  practically identifiable: {res.practically_identifiable}')
print(f"{'param':<10}{'central':>10}{'pred RSE %':>12}{'IIV CV %':>10}  identifiable?")
for p in res.params:
    cv = '-' if p.iiv_cv_percent is None else f'{p.iiv_cv_percent:.0f}'
    print(f'{p.symbol:<10}{p.central:>10.4g}{p.rse_percent:>12.0f}{cv:>10}  {p.identifiable}')

## CV is not identifiability

The kill rate `kD` is well identified (its early-shrinkage signal is strong), but the growth rate `kL` and the resistance decay `lambda` are flat under a short trial — so `lambda`'s ~90% CV is, at least in part, a flat-likelihood artifact of the design, not a clean estimate of biological spread. That is exactly the parameter whose out-of-context transport is least defensible.

In [ ]:
for w in res.warnings:
    print('!', w)

## Identifiability is a property of the *design*: lengthen the follow-up

A superset schedule can only add information (a positive-semidefinite Fisher-information increment), so no parameter's predicted RSE can rise. `kD` is identifiable from any short trial; `lambda` only drops below the 50% ceiling once follow-up runs past resistance-driven regrowth.

In [ ]:
order = ['kL', 'kD', 'lambda']
horizons = [24, 36, 48, 72, 104, 156]
rows = {}
for h in horizons:
    r = onkos.identifiability(ds, 'resistance.claret_2009.tgi', context=ctx,
                              schedule=list(np.arange(0.0, h + 1e-9, 6.0)))
    b = {p.symbol: p.rse_percent for p in r.params}
    rows[h] = [b[s] for s in order]

print(f"{'horizon wk':>10}" + ''.join(f'{s:>12}' for s in order))
for h in horizons:
    print(f'{h:>10}' + ''.join(f'{v:>12.0f}' for v in rows[h]))

# Monotonic information: a longer schedule never worsens precision.
assert all(rows[156][j] <= rows[24][j] + 1e-6 for j in range(len(order)))

## Curation triage: which models can a realistic trial support?

The report-level summary ranks clinical TGI models by whether the reference design can estimate them at all. The parsimonious 2-parameter biexponential (Stein/Bruno) models are identifiable; the 3-parameter resistance-augmented models are not — the resistance/capacity terms are the culprits.

In [ ]:
from onkos.report import practical_identifiability_summary
for m in practical_identifiability_summary(ds):
    verdict = 'identifiable' if m['identifiable'] else f"NOT (worst: {m['worst']})"
    print(f"{m['record_id']:<48} {m['n_params']}p  {verdict}")

## Guardrails: a design diagnostic, not a tier and not a prediction

Identifiability cannot move a tier (it passes the record's propagated tier through), a singular design reports `inf` rather than a fabricated bound, and the result carries the clinical-use prohibition with its named design. "Unidentifiable" always means *under this schedule and residual-error model*.

In [ ]:
# Tier passes through unchanged; an out-of-context transport still floors to D.
assert res.tier == ds['resistance.claret_2009.tgi'].tier
out = onkos.identifiability(ds, 'resistance.claret_2009.tgi',
                            context={'tumor_type': 'melanoma', 'line': 'first'})
assert out.tier == 'D'

d = res.to_dict()
assert d['NOT_FOR_CLINICAL_USE'] is True and 'PROHIBITED' in d['onkos:clinicalUse']
assert 'schedule_weeks' in d['design'] and 'practically_identifiable' in d
print('guardrails OK | transported tier:', out.tier, '| verdict travels with design:',
      d['practically_identifiable'])